# SolarKG-VN: chạy toàn bộ thực nghiệm trên Kaggle

**Settings cần bật trước khi chạy:** Accelerator = **GPU (T4 / P100)**, **Internet = On**.

Thứ tự các bước:
1. Lấy code và cài thư viện
2. Thực nghiệm chính: Bảng 7–10, Hình 1–3
3. Benchmark chuẩn UMLS / Kinship / Nations (R2-7, R2-8)
4. Đo thời gian chạy và khả năng mở rộng (R1-B, R2-4)
5. Baseline LLM và trích xuất bằng LLM với DeepSeek V4.1 Flash qua API (R1-A, R2-6)

Kết quả được copy vào `/kaggle/working/results` sau mỗi bước, nên phiên có bị ngắt giữa chừng thì vẫn giữ được những bước đã xong.
Bước nào chạy rồi thì đặt cờ tương ứng ở cell cấu hình thành `False` để bỏ qua.

In [ ]:
# Cấu hình: bật/tắt từng bước
RUN_MAIN        = True   # ~1 giờ
RUN_BENCHMARKS  = True   # ~1-2 giờ
RUN_SCALABILITY = True   # ~30-60 phút
RUN_LLM         = True   # DeepSeek V4.1 Flash qua API (cần secret DEEPSEEK_API_KEY)
DEEPSEEK_MODEL  = "deepseek-ai/DeepSeek-V4.1-Flash-1017"
DEEPSEEK_BASE_URL = "https://inference.14092005.xyz/openai/v1"
DEEPSEEK_EXTRA_BODY = '{"enable_thinking": false}'   # tắt chế độ thinking
LLM_MAX_DOCS    = 500    # số tài liệu dùng để đánh giá trích xuất

## 1. Lấy code và cài thư viện

In [ ]:
import os, subprocess, shutil, time
if not os.path.exists("/kaggle/working/ai-knowledge-reasoning"):
    !git clone https://github.com/McGarrett7/ai-knowledge-reasoning.git /kaggle/working/ai-knowledge-reasoning
%cd /kaggle/working/ai-knowledge-reasoning
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
!nvidia-smi --query-gpu=name,memory.total --format=csv || true
!python -m unittest discover -s tests

In [ ]:
OUT = "/kaggle/working/results"
def save():
    shutil.copytree("results", OUT, dirs_exist_ok=True)
    print("saved ->", OUT)

## 2. Thực nghiệm chính (5 seed): Bảng 7–10, các hình

In [ ]:
if RUN_MAIN:
    !python main.py run 2>&1 | tee results/run_main.log
    save()

## 3. Benchmark chuẩn: UMLS, Kinship, Nations (3 seed)

In [ ]:
if RUN_BENCHMARKS:
    !python main.py benchmarks 2>&1 | tee results/run_benchmarks.log
    save()

## 4. Đo thời gian chạy và khả năng mở rộng
Nên chạy bước này khi không có tác vụ nặng nào khác chạy song song, để số đo thời gian không bị lệch.

In [ ]:
if RUN_SCALABILITY:
    !python main.py scalability 2>&1 | tee results/run_scalability.log
    save()

## 5. LLM: DeepSeek V4.1 Flash qua API (R1-A, R2-6)
Trước khi chạy, thêm key vào **Add-ons → Secrets**, đặt tên `DEEPSEEK_API_KEY`, rồi bật quyền truy cập cho notebook.
Không dán key trực tiếp vào notebook. Câu trả lời của LLM được cache trong `results/llm_cache/`, nên chạy lại không bị tính phí lần nữa.

In [ ]:
if RUN_LLM:
    from kaggle_secrets import UserSecretsClient
    os.environ["DEEPSEEK_API_KEY"] = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
    os.environ["DEEPSEEK_BASE_URL"] = DEEPSEEK_BASE_URL
    os.environ["DEEPSEEK_EXTRA_BODY"] = DEEPSEEK_EXTRA_BODY
    # R1-A: zero-shot và KG-RAG, có và không có lớp paraconsistent
    !python main.py llm --backend deepseek --model {DEEPSEEK_MODEL} 2>&1 | tee results/run_llm.log
    # R2-6: chất lượng trích xuất triple bằng LLM
    !python main.py extract --backend deepseek --model {DEEPSEEK_MODEL} --max-docs {LLM_MAX_DOCS} 2>&1 | tee results/run_extract.log
    save()

## 6. Xem nhanh và tải kết quả về

In [ ]:
from IPython.display import Image, Markdown, display
if os.path.exists("results/tables.md"):
    display(Markdown(open("results/tables.md", encoding="utf-8").read()))
for f in sorted(os.listdir("results/figures")) if os.path.exists("results/figures") else []:
    print(f); display(Image(f"results/figures/{f}", width=650))
shutil.make_archive("/kaggle/working/results", "zip", OUT)
print("Tải file /kaggle/working/results.zip ở tab Output")